In [54]:
# Importing libraries for the GOOGL stock price prediction, using
# Multiple linear regression and Gradient boosting.

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import root_mean_squared_error

In [55]:
# Loading and preparing the dataset:

df = pd.read_csv("alphabet_prepared.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

In [56]:
# Creating the target variable (which is the next-day closing price)
df['y_next'] = df['Price'].shift(-1)

In [57]:
# Selecting features for H1, including lagged prices only:

features_A = ['Price_Yesterday', 'Price_5_DaysAgo'] 

# Selecting features for H2 and H3, which are the lagged proces and the technical indicators:

features_B = features_A + [
    'MovingAvg_10day','MovingAvg_20day','MovingAvg_50day','MovingAvg_200day',
    'RSI','MACD','MACD_SignalLine','Volatility_20day','Vol.'
]

# The MovingAvg_10day is the short-term average, the MovingAvg_20day is the medium-term average.
# MovingAvg_50day is a medium/long-term average, and MovingAvg_200day is a long-term average.


In [58]:
# Dropping rows with missing/null values:

df = df.dropna(subset=features_B + ['y_next'])

X_A = df[features_A]   # for model A
X_B = df[features_B]   # for model B and gradient booting
y = df['y_next']     # target (next day closing price)
dates = df['Date'] 

In [59]:
# Separating data chronologically:

train_idx = dates < "2024-01-01"   # our training data (2021-2023)
valid_idx = (dates >= "2024-01-01") & (dates < "2025-01-01") # just a validation data (2024)
test_idx  = dates >= "2025-01-01"   # our testing data (2025)

In [60]:
# Calculating evaluation metrics:

def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "RMSE": root_mean_squared_error(y_true, y_pred),  # this will be our avg predictor error
        "MAE": mean_absolute_error(y_true, y_pred),    # error in monetary value
        "R2": r2_score(y_true, y_pred)    # % of the variance explained
    }

results = []

In [61]:
'Multiple Linear Regression'

'Multiple Linear Regression'

In [62]:
# Model A (H1):

pipe_lr_A = Pipeline([
    ("scaler", StandardScaler()),      # normalizing data so we can compare them properly
    ("lr", LinearRegression())    
])
pipe_lr_A.fit(X_A[train_idx], y[train_idx])
results.append(evaluate_model("LR_A (lags) - Valid", y[valid_idx], pipe_lr_A.predict(X_A[valid_idx])))
results.append(evaluate_model("LR_A (lags) - Test",  y[test_idx],  pipe_lr_A.predict(X_A[test_idx])))

In [63]:
# Model B (H2):

pipe_lr_B = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])
pipe_lr_B.fit(X_B[train_idx], y[train_idx])
results.append(evaluate_model("LR_B (lags+tech) - Valid", y[valid_idx], pipe_lr_B.predict(X_B[valid_idx])))
results.append(evaluate_model("LR_B (lags+tech) - Test",  y[test_idx],  pipe_lr_B.predict(X_B[test_idx])))

In [64]:
# Most relevant coefficients obtained from Model B:

coef_B = pd.Series(pipe_lr_B.named_steps['lr'].coef_, index=features_B).sort_values(key=abs, ascending=False)
print("\nTop 10 Linear Regression Coefficients (Model B):")
print(coef_B.head(10))


Top 10 Linear Regression Coefficients (Model B):
MA20            27.974474
Signal         -13.993919
MACD            13.940659
MA50            -8.283159
Price_lag1      -6.086593
MA10             5.986386
Price_lag5      -2.353964
RSI              1.400088
MA200           -1.109558
Volatility20    -0.049107
dtype: float64


In [65]:
# With the previous output we have the expected change in the target variable (next day closing price)
# when there is a increase of 1 unit on the predictor, while keeping other variables constant.

# MA20 shows that for every 1 unit increase in the 20 day moving avg, the predicted target will increase by 27.97 units.

# Signal shows that for every 1 unit increase in signal, the predictor will decrease by 13.99 units. 

# We can see that the most influential coefficients are MA20, Signal and also MACD, which are larger compared to the others.

In [66]:
'Gradient Boosting (machine learning)'

'Gradient Boosting (machine learning)'

In [67]:
# Model C (H3):

gb = GradientBoostingRegressor(random_state=42)

# Adjusting the parameters:

param_grid = {
    "n_estimators": [200, 400],    # number of trees
    "max_depth": [2, 3],   
    "learning_rate": [0.05, 0.1]
}


grid = GridSearchCV(
    gb,
    param_grid,
    cv=[(train_idx, valid_idx)],  # train/valid split
    scoring="neg_root_mean_squared_error"
)
grid.fit(X_B, y)

# Best gradient boosting model:

best_gb = grid.best_estimator_
results.append(evaluate_model("GB - Valid", y[valid_idx], best_gb.predict(X_B[valid_idx])))
results.append(evaluate_model("GB - Test",  y[test_idx],  best_gb.predict(X_B[test_idx])))

In [68]:
# Showing feature importances from the gradient boosting model:

gb_importance = pd.Series(best_gb.feature_importances_, index=features_B).sort_values(ascending=False)
print("\nTop 10 Gradient Boosting Feature Importances:")
print(gb_importance.head(10))


Top 10 Gradient Boosting Feature Importances:
Price_lag1    0.858529
MA10          0.111801
MA20          0.016880
RSI           0.004010
Price_lag5    0.003303
MA50          0.002083
Vol.          0.000971
MA200         0.000955
Signal        0.000863
MACD          0.000339
dtype: float64


In [69]:
# The previuos output shows how much each feature contributes to reducing error in the ensemble trees.

#Price_lag1 is the main feature, with 85.85% importance. 
# It indicates that the previous day price (Price_lag1) is the most predictive feature in this model.

# When analysing data with this model we can get the dominant feature, even if the linear coefficient was smaller, for example.

In [70]:
# Final results table:

results_df = pd.DataFrame(results)
print("\nModel Evaluation Results:")
print(results_df)


Model Evaluation Results:
                      Model      RMSE       MAE        R2
0       LR_A (lags) - Valid  4.287740  3.262889  0.922010
1        LR_A (lags) - Test  4.865704  3.969833  0.888093
2  LR_B (lags+tech) - Valid  3.944649  3.032682  0.933992
3   LR_B (lags+tech) - Test  5.288715  4.386277  0.867789
4                GB - Valid  2.627181  1.948461  0.970721
5                 GB - Test  2.312633  1.837516  0.974720


In [71]:
# In general, there is a great proportion of variance being explained, as showed by the high R2s.

# By adding technical indicators on LR_B the performance increased a bit, but decreased RMSE as well.
# The opposite happened to the test performance. Can indicate overfitting.

# The gradient boosting model performs better than the previous models, with less errors and higher R2s.

# In the end, the gradient boosting test seems to be more robust, capturing the nonlinear patterns that the LR may miss.